In [13]:
import xarray as xr
import numpy as np

from datetime import datetime, UTC

# We define some helper functions:
def td_to_iso(td):
    # Computes np.timedelta64 to ISO8601 standard
    # cannot handle variable-length units such as months and years
    
    nanoseconds = td.astype("timedelta64[ns]").astype(int)
    
    seconds = nanoseconds/1e9
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    days, hours = divmod(hours, 24)

    return f"P{int(days)}D{int(hours)}H{int(minutes)}M{seconds}S"

def ts_to_iso(ts):
    #converts a np.datetime64 timestamp to the ISO 8601 format
    return str(ts).replace("-","").replace(":","")

def maxdelta(ds, coord):
    # computes the maximum delta for a dataset and a specified coordinate
    return np.max(abs(ds[coord][1:] - ds[coord][:-1]).values)


def get_ds(ds_in):

    """ Function to create a netCDF/CF and ACDD compliant dataset using xarray, such that results can be written to disk. 

    Args:
        ds_in (xr.Dataset/xr.DataArray)

    Returns:
        ds_out (xr.Dataset/xr.DataArray)
        
    """

    ds_out = ds_in.copy(deep=True).drop_vars(("x", "y"))
    # we drop x, y variables since they are redundant

    # we add general metadata that is specific to the example dataset
    ds_out.attrs["title"] = "KaRIn SSH"
    ds_out.attrs["Conventions"] = "CF-1.3, ACDD-1.3"
    ds_out.attrs["source"] = "Satellite observations using the Ka-band Radar Interferometer (KaRIn)"

    # Add history
    old_history = ds_out.attrs.get("history", "")
    entry = f"{datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")} - Converted dataset to be CF-1.13 compliant"

    if old_history:
        ds_out.attrs["history"] = ds_out.attrs.get("history", "")
    else:
        ds_out.attrs["history"] = old_history + "\n" + entry


    # we fill in standard_name where it is missing
    ds_out["mean_sea_surface_cnescls"].attrs["standard_name"] = "sea_surface_height_above_reference_ellipsoid"
    ds_out["bathy"].attrs["standard_name"] = "N/A"
    ds_out["eta_detrend"].attrs["standard_name"] = "N/A"

    # we mark some of the variables as ancillary variables
    ds_out["ssh_karin_2"].attrs["ancillary_variables"] = "ssh_karin_uncert"
    ds_out["sig0_karin_2"].attrs["ancillary_variables"] = "sig0_karin_uncert"


    # we compute the latitude and longitude range of the data
    ds_out.attrs["geospatial_lat_min"] = ds_out.coords["latitude"].min().item()
    ds_out.attrs["geospatial_lat_max"] = ds_out.coords["latitude"].max().item()

    ds_out.attrs["geospatial_lon_min"] = ds_out.coords["longitude"].min().item()
    ds_out.attrs["geospatial_lon_max"] = ds_out.coords["longitude"].max().item()


    # We compute time-related metadata
    # Start/stop time, duration and resolution needs to follow the ISO-8601 standard
    # Firstly, when does it start and when does it end:
    ds_out.attrs["time_coverage_start"] = ts_to_iso(np.min(ds_in["time"].values))
    ds_out.attrs["time_coverage_end"] = ts_to_iso(np.max(ds_in["time"].values))

    # Secondly, duration
    ds_out.attrs["time_coverage_duration"] = td_to_iso(np.max(ds_in["time"].values)-np.min(ds_in["time"].values))

    # Thirdly, infer the resolution
    ds_out.attrs["time_coverage_resolution"] = td_to_iso(maxdelta(ds_out, "time"))

    # Fourthly, add time at which it was last modified
    ds_out.attrs["date_modified"] = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    
    return ds_out


In [14]:
ds_ss=xr.open_dataset('../ds_ss.nc')

ds_ss

<xarray.Dataset> Size: 52MB
Dimensions:                                (num_lines: 2600, num_pixels: 173)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 4MB ...
    longitude                              (num_lines, num_pixels) float64 4MB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/20)
    time                                   (num_lines) datetime64[ns] 21kB ...
    time_tai                               (num_lines) datetime64[ns] 21kB ...
    latitude_uncert                        (num_lines, num_pixels) float64 4MB ...
    longitude_uncert                       (num_lines, num_pixels) float64 4MB ...
    polarization_karin                     (num_lines) object 21kB ...
    ssh_karin_2                            (num_lines, num_pixels) float64 4MB ...
    ...                                     ...
    miti_power_var_250m                    (num_lines, num_pixels) float32 2MB ...
    ancillary_surface_classification_flag  (num_lines, num_pixels) float32 2MB ...
    bathy                                  (num_lines, num_pixels) float64 4MB ...
    eta_detrend                            (num_lines, num_pixels) float64 4MB ...
    x                                      (num_pixels) float64 1kB ...
    y                                      (num_lines) float64 21kB ...
Attributes:
    description:  Unsmoothed SSH measurement data and related information for...

In [15]:
ds_ss_formatted = get_ds(ds_ss)
ds_ss_formatted

<xarray.Dataset> Size: 52MB
Dimensions:                                (num_lines: 2600, num_pixels: 173)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 4MB ...
    longitude                              (num_lines, num_pixels) float64 4MB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/18)
    time                                   (num_lines) datetime64[ns] 21kB ...
    time_tai                               (num_lines) datetime64[ns] 21kB ...
    latitude_uncert                        (num_lines, num_pixels) float64 4MB ...
    longitude_uncert                       (num_lines, num_pixels) float64 4MB ...
    polarization_karin                     (num_lines) object 21kB ...
    ssh_karin_2                            (num_lines, num_pixels) float64 4MB ...
    ...                                     ...
    mean_sea_surface_cnescls               (num_lines, num_pixels) float64 4MB ...
    miti_power_250m                        (num_lines, num_pixels) float32 2MB ...
    miti_power_var_250m                    (num_lines, num_pixels) float32 2MB ...
    ancillary_surface_classification_flag  (num_lines, num_pixels) float32 2MB ...
    bathy                                  (num_lines, num_pixels) float64 4MB ...
    eta_detrend                            (num_lines, num_pixels) float64 4MB ...
Attributes: (12/14)
    description:               Unsmoothed SSH measurement data and related in...
    title:                     KaRIn SSH
    Conventions:               CF-1.3, ACDD-1.3
    source:                    Satellite observations using the Ka-band Radar...
    history:                   \n20260630T115045Z - Converted dataset to be C...
    geospatial_lat_min:        54.544354
    ...                        ...
    geospatial_lon_max:        219.53052699999998
    time_coverage_start:       20231017T122727.288529536
    time_coverage_end:         20231017T122903.820543232
    time_coverage_duration:    P0D0H1M36.53201369600001S
    time_coverage_resolution:  P0D0H0M0.03714368S
    date_modified:             20260630T115045Z